# Train the KOI planet-candidate classifier

This notebook downloads the **NASA Exoplanet Archive KOI Cumulative Delivery** through TAP, turns `CONFIRMED` + `CANDIDATE` into the planet-like class and `FALSE POSITIVE` into the negative class, trains a calibrated gradient-boosted tree model, evaluates it on a star-grouped holdout set, and exports a runtime-ready model bundle.

The chosen features match values the transit pipeline can produce: period, duration, depth, SNR, number of observed transits, and optional stellar radius. Archive disposition flags and `koi_score` are intentionally excluded because they leak the label.

Data documentation: [NASA TAP guide](https://exoplanetarchive.ipac.caltech.edu/docs/TAP/usingTAP.html) · [KOI column definitions](https://exoplanetarchive.ipac.caltech.edu/docs/API_kepcandidate_columns.html)

## 1. Setup

From the repository root, install the notebook environment once with:

```powershell
.venv\Scripts\python.exe -m pip install -r notebooks/requirements.txt
.venv\Scripts\python.exe -m jupyter lab
```

In [ ]:
from __future__ import annotations

import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlencode

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    brier_score_loss,
    classification_report,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline

pd.set_option('display.max_columns', 50)
plt.style.use('seaborn-v0_8-darkgrid')

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data'
MODEL_DIR = PROJECT_ROOT / 'models'
DATA_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
REFRESH_DATA = False  # Set True when you explicitly want a fresh NASA download.
print({'python': platform.python_version(), 'pandas': pd.__version__, 'scikit_learn': sklearn.__version__, 'project_root': str(PROJECT_ROOT)})

## 2. Download and cache labeled KOIs

The first run downloads the selected columns. Later runs use `data/koi_cumulative_training.csv` unless `REFRESH_DATA` is changed to `True`. The cumulative table is a useful hackathon label source, but NASA notes that it combines multiple deliveries and is not a uniform statistical population.

In [ ]:
TAP_ENDPOINT = 'https://exoplanetarchive.ipac.caltech.edu/TAP/sync'
RAW_COLUMNS = [
    'kepid',
    'kepoi_name',
    'koi_disposition',
    'koi_period',
    'koi_duration',
    'koi_depth',
    'koi_model_snr',
    'koi_num_transits',
    'koi_srad',
]
COLUMN_MAP = {
    'koi_period': 'period_days',
    'koi_duration': 'duration_hours',
    'koi_depth': 'depth_ppm',
    'koi_model_snr': 'snr',
    'koi_num_transits': 'num_transits_observed',
    'koi_srad': 'stellar_radius_solar',
}
FEATURE_COLUMNS = list(COLUMN_MAP.values())
CACHE_PATH = DATA_DIR / 'koi_cumulative_training.csv'

TAP_QUERY = f"""
SELECT {','.join(RAW_COLUMNS)}
FROM cumulative
WHERE koi_disposition IN ('CONFIRMED','CANDIDATE','FALSE POSITIVE')
""".strip()
TAP_URL = f"{TAP_ENDPOINT}?{urlencode({'query': ' '.join(TAP_QUERY.split()), 'format': 'csv'})}"

if REFRESH_DATA or not CACHE_PATH.exists():
    print('Downloading labeled KOIs from NASA Exoplanet Archive...')
    raw = pd.read_csv(TAP_URL)
    if raw.empty:
        raise RuntimeError('NASA TAP returned an empty table.')
    raw.to_csv(CACHE_PATH, index=False)
    print(f'Cached {len(raw):,} rows at {CACHE_PATH}')
else:
    raw = pd.read_csv(CACHE_PATH)
    print(f'Loaded {len(raw):,} cached rows from {CACHE_PATH}')

missing_columns = sorted(set(RAW_COLUMNS) - set(raw.columns))
if missing_columns:
    raise ValueError(f'NASA schema check failed; missing columns: {missing_columns}')
display(raw.head(3))

## 3. Audit labels and prepare features

This is a **weak-label** setup: `CANDIDATE` is grouped with `CONFIRMED`, exactly as proposed in the architecture, but a candidate is not guaranteed to be a real planet.

In [ ]:
df = raw.copy()
df = df.drop_duplicates(subset='kepoi_name', keep='last')
df['label'] = df['koi_disposition'].isin(['CONFIRMED', 'CANDIDATE']).astype('int8')
df = df.rename(columns=COLUMN_MAP)
df[FEATURE_COLUMNS] = df[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)

# Remove impossible measurements, but let the model pipeline impute ordinary missing values.
for column in ['period_days', 'duration_hours', 'depth_ppm', 'snr', 'num_transits_observed', 'stellar_radius_solar']:
    df.loc[df[column] <= 0, column] = np.nan
df = df[df[FEATURE_COLUMNS].notna().sum(axis=1) >= 4].copy()

X = df[FEATURE_COLUMNS]
y = df['label']
groups = df['kepid'].fillna(df['kepoi_name']).astype(str)

label_audit = pd.DataFrame({
    'rows': df['koi_disposition'].value_counts(),
    'share': df['koi_disposition'].value_counts(normalize=True).round(4),
})
display(label_audit)
display(X.describe(percentiles=[.01, .5, .99]).T)
print(f'Training-eligible rows: {len(df):,}; unique host groups: {groups.nunique():,}; missing cells: {X.isna().sum().sum():,}')

## 4. Make star-grouped train, validation, and test splits

All KOIs belonging to the same Kepler star stay in one split. That prevents the model from being evaluated on a sibling signal from a host it saw during training.

In [ ]:
outer_split = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_full_idx, test_idx = next(outer_split.split(X, y, groups=groups))
X_train_full, X_test = X.iloc[train_full_idx], X.iloc[test_idx]
y_train_full, y_test = y.iloc[train_full_idx], y.iloc[test_idx]
groups_train_full = groups.iloc[train_full_idx]

inner_split = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE + 1)
train_idx, validation_idx = next(inner_split.split(X_train_full, y_train_full, groups=groups_train_full))
X_train, X_validation = X_train_full.iloc[train_idx], X_train_full.iloc[validation_idx]
y_train, y_validation = y_train_full.iloc[train_idx], y_train_full.iloc[validation_idx]

split_summary = pd.DataFrame({
    'rows': [len(X_train), len(X_validation), len(X_test)],
    'planet_like_share': [y_train.mean(), y_validation.mean(), y_test.mean()],
}, index=['train', 'validation', 'test'])
display(split_summary.style.format({'planet_like_share': '{:.1%}'}))
assert set(groups.iloc[test_idx]).isdisjoint(set(groups_train_full)), 'Host leakage into the test split.'
assert y_train.nunique() == y_validation.nunique() == y_test.nunique() == 2, 'Every split must contain both labels.'

## 5. Train and calibrate gradient-boosted trees

Median imputation is learned from training folds only. The sigmoid calibration layer makes the output more useful as a ranking confidence, although it still must not be presented as a formal planet probability.

In [ ]:
base_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('classifier', HistGradientBoostingClassifier(
        learning_rate=0.055,
        max_iter=240,
        max_leaf_nodes=31,
        min_samples_leaf=24,
        l2_regularization=1.0,
        class_weight='balanced',
        random_state=RANDOM_STATE,
    )),
])
model = CalibratedClassifierCV(base_model, method='sigmoid', cv=5, n_jobs=1)
model.fit(X_train, y_train)
print('Model fitted:', model)

## 6. Choose the operating threshold on validation data

The probability model is never tuned on the final test set. Here the decision threshold is selected for the best validation F1 score; the dashboard can still display the continuous ranking score.

In [ ]:
validation_probability = model.predict_proba(X_validation)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_validation, validation_probability)
f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
best_index = int(np.nanargmax(f1))
decision_threshold = float(thresholds[best_index])
print({
    'decision_threshold': round(decision_threshold, 4),
    'validation_f1': round(float(f1[best_index]), 4),
    'validation_precision': round(float(precision[best_index]), 4),
    'validation_recall': round(float(recall[best_index]), 4),
})

## 7. Evaluate once on the untouched test set

In [ ]:
test_probability = model.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= decision_threshold).astype(int)
test_metrics = {
    'roc_auc': float(roc_auc_score(y_test, test_probability)),
    'average_precision': float(average_precision_score(y_test, test_probability)),
    'brier_score': float(brier_score_loss(y_test, test_probability)),
}
display(pd.Series(test_metrics, name='test').round(4).to_frame())
report = pd.DataFrame(classification_report(
    y_test,
    test_prediction,
    labels=[0, 1],
    target_names=['false positive', 'planet-like'],
    output_dict=True,
    zero_division=0,
)).T
display(report.round(3))

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay.from_predictions(
    y_test, test_prediction, display_labels=['false positive', 'planet-like'], cmap='Blues', ax=axes[0], colorbar=False
)
axes[0].set_title(f'Grouped holdout confusion matrix\nthreshold = {decision_threshold:.3f}')
CalibrationDisplay.from_predictions(y_test, test_probability, n_bins=10, strategy='quantile', ax=axes[1])
axes[1].set_title('Holdout calibration')
figure.tight_layout()
plt.show()

## 8. Inspect feature importance

Permutation importance measures the drop in holdout ROC AUC when each feature is shuffled. It is model behavior, not a claim of physical causality.

In [ ]:
importance = permutation_importance(
    model, X_test, y_test, scoring='roc_auc', n_repeats=8, random_state=RANDOM_STATE, n_jobs=1
)
importance_table = pd.DataFrame({
    'feature': FEATURE_COLUMNS,
    'mean_auc_drop': importance.importances_mean,
    'std': importance.importances_std,
}).sort_values('mean_auc_drop', ascending=False)
display(importance_table.style.format({'mean_auc_drop': '{:.4f}', 'std': '{:.4f}'}))
ax = importance_table.sort_values('mean_auc_drop').plot.barh(
    x='feature', y='mean_auc_drop', xerr='std', legend=False, figsize=(8, 4), color='#4f8cc9'
)
ax.set_xlabel('Mean decrease in test ROC AUC')
ax.set_ylabel('')
ax.set_title('Permutation importance')
plt.tight_layout()
plt.show()

## 9. Export the trained model and feature contract

The `.joblib` file contains the fitted imputer, calibrated classifier, feature order, threshold, source query, and metrics. Only load joblib artifacts you created or trust.

In [ ]:
trained_at = datetime.now(timezone.utc).isoformat()
model_path = MODEL_DIR / 'koi_candidate_classifier.joblib'
metadata_path = MODEL_DIR / 'koi_candidate_classifier.metadata.json'

metadata = {
    'model_name': 'nasa-koi-hist-gradient-boosting-v1',
    'trained_at_utc': trained_at,
    'training_rows': int(len(X_train)),
    'validation_rows': int(len(X_validation)),
    'test_rows': int(len(X_test)),
    'feature_columns': FEATURE_COLUMNS,
    'source_table': 'NASA Exoplanet Archive cumulative KOI table',
    'source_endpoint': TAP_ENDPOINT,
    'source_query': TAP_QUERY,
    'label_definition': {'planet_like': ['CONFIRMED', 'CANDIDATE'], 'false_positive': ['FALSE POSITIVE']},
    'decision_threshold': decision_threshold,
    'test_metrics': test_metrics,
    'versions': {'python': platform.python_version(), 'pandas': pd.__version__, 'scikit_learn': sklearn.__version__},
}
bundle = {'model': model, **metadata}
joblib.dump(bundle, model_path)
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print(f'Saved model: {model_path} ({model_path.stat().st_size / 1_000_000:.2f} MB)')
print(f'Saved metadata: {metadata_path}')

## 10. Reload smoke test

This final cell proves the saved bundle can make predictions in a fresh load path.

In [ ]:
loaded = joblib.load(model_path)
sample = X_test.head(5).copy()
sample_scores = loaded['model'].predict_proba(sample)[:, 1]
smoke_result = sample.assign(planet_like_score=sample_scores)
display(smoke_result)
assert np.isfinite(sample_scores).all() and ((0 <= sample_scores) & (sample_scores <= 1)).all()
print('Reload smoke test passed.')

## What this model does—and does not—mean

- It is trained on completed Kepler KOI labels, not the synthetic dashboard fixtures.
- It ranks how similar a candidate's catalog-level measurements are to planet-like KOIs. It does not confirm planets.
- `CANDIDATE` is treated as positive for hackathon scope, so the target is intentionally noisy.
- The cumulative table combines multiple deliveries; do not use this notebook for occurrence-rate or population studies.
- TESS deployment would need mission-specific validation or retraining because its cadence and noise differ from Kepler.
- Odd/even and secondary-eclipse measurements remain runtime vetting features. They are excluded here because equivalent continuous columns are not available in the selected cumulative-table contract.